## tl;dr
2026-09-05 18:09:27 JSTの固定スナップショット。改善ラウンドの完了6モデルは追加70,722,171トークン、API標準料金換算$45.57268184。Fableは進行中なので合計に含めない。

初回提出物・既存評価・ログは変更していない。

## Context & Methods
改善ラウンドの時間、トークン、API標準料金換算を復元する監査用の付属ノートブック。

### Key Assumptions
- 時間は関連ターンの開始→完了の経過時間。ツール実行を含み、ターン間のユーザー待ち時間を除外。最初の中断分は`feedback_prep`に分け、`feedback_all`に含む。
- 初回は既存の確定済み本実行ターンを維持。初回準備・終了後の質問は除外。
- トークンは応答IDで重複除去。キャッシュ入力を含み、reasoningはoutputの内数。Opusの中断中1応答だけreasoning内訳が不明だが総出力3トークンは記録済み。
- Codexは応答ごとの使用量と全てのturn/thread累積値を照合。従来のUI `token_count`だけでは今回の圧縮処理を一部落とすため採用しない。
- Claudeはmessage.idをキーに重複除去しrequestIdも照合。全キャッシュ書込はログ上1時間TTLなので2倍入力単価を適用。
- 料金は2026-09-05確認の公開API Standard価格。実際の請求額・定額契約費ではない。Codexの実サービス階層はログから確定不可。Fastならこの換算額の2倍を別シナリオに保存。税・為替・地域加算・契約割引・ローカル計算費は含まない。
- GPTの1リクエスト272K入力超割増を実装したが、対象には該当なし。Claudeは今回の全モデルで長文割増なし。
- Fableは指定時点までに記録された応答のみ。実行中リクエストの未記録使用量を含まない。

料金根拠：[OpenAI](https://developers.openai.com/api/docs/pricing)、[Anthropic](https://platform.claude.com/docs/en/about-claude/pricing)。Sonnet 5は当初予定の9月値上げが撤回され$2/$10が標準料金。Fable 5.1のcache-readは$0.25/MTok。

## Data
`usage_cost_snapshot.json`に元ログ7本の絶対パス・バイト数・SHA-256・集計時点を保存。`recover_usage_cost.py`が抽出、重複除去、料金計算を担当し、ファイル書込は行わない。

再実行は保存済みログのprefixだけを読み、Fableに追記があっても同じ結果を再現する。

In [1]:
import sys, json
from pathlib import Path
BASE = Path('/Users/ankimo1210/Documents/projects/quant-agent-benchmark/analysis/feedback-round-01-usage-20260905')
sys.path.insert(0, str(BASE))
import recover_usage_cost as recovery
snapshot = json.loads((BASE / 'usage_cost_snapshot.json').read_text())


### 1. 元ログから再計算して独立照合

In [2]:
recomputed = recovery.recover(snapshot)
assert recomputed == snapshot
for item in recomputed['summaries']:
    initial, whole, prep, main = [item[k] for k in ['initial','feedback_all','feedback_prep','feedback_main']]
    assert whole['total_tokens'] == prep['total_tokens'] + main['total_tokens']
    assert abs(whole['usd_standard'] - prep['usd_standard'] - main['usd_standard']) < 1e-9
    rates = recovery.RATES[item['model_ids'][0]]
    independent_cost = sum(whole[k] * rate for k, rate in zip(['uncached_input','cache_read_input','cache_write_5m','cache_write_1h','output_total'], rates)) / 1e6
    assert whole['long_context_requests'] == 0
    assert abs(independent_cost - whole['usd_standard']) < 1e-9
print('PASS: source hashes, response deduplication, Codex turn/thread counters, initial baseline, phase sums, independent pricing, exact snapshot replay.')


PASS: source hashes, response deduplication, Codex turn/thread counters, initial baseline, phase sums, independent pricing, exact snapshot replay.


## Results
時間は分、tokenはキャッシュ込総処理量、料金はUSD Standard換算。

In [3]:
print('model\tstatus\tminutes\ttotal_tokens\tUSD\tcache_read_pct')
for item in recomputed['summaries']:
    value = item['feedback_all']
    print(f"{item['model']}\t{value['status']}\t{value['work_minutes']:.2f}\t{value['total_tokens']:,}\t{value['usd_standard']:.2f}\t{100*value['cache_read_share']:.2f}%")
completed = [x['feedback_all'] for x in recomputed['summaries'] if x['feedback_all']['status'] == 'complete']
print(f"Completed six: {sum(x['total_tokens'] for x in completed):,} tokens; ${sum(x['usd_standard'] for x in completed):.2f}")


model	status	minutes	total_tokens	USD	cache_read_pct
astra	complete	42.80	3,953,530	11.34	87.48%
sol	complete	23.14	8,698,755	6.34	94.12%
terra	complete	24.56	8,940,236	3.33	94.86%
luna	complete	24.26	6,745,535	0.26	94.51%
sonnet	complete	39.31	18,429,565	6.84	96.84%
opus	complete	27.81	23,954,550	17.45	98.43%
fable	running	26.04	6,110,461	17.93	89.87%
Completed six: 70,722,171 tokens; $45.57


### 2. 初回＋改善の累計
準備・中断分のうち改善に直接対応するもののみ改善累計へ追加。Fableは途中値。

In [4]:
print('model\tinitial_minutes\textra_minutes\tcumulative_minutes\tcumulative_tokens\tcumulative_USD')
for item in recomputed['summaries']:
    old, new = item['initial'], item['feedback_all']
    print(f"{item['model']}\t{old['work_minutes']:.2f}\t{new['work_minutes']:.2f}\t{old['work_minutes']+new['work_minutes']:.2f}\t{old['total_tokens']+new['total_tokens']:,}\t{old['usd_standard']+new['usd_standard']:.2f}")


model	initial_minutes	extra_minutes	cumulative_minutes	cumulative_tokens	cumulative_USD
astra	51.97	42.80	94.78	7,751,222	20.41
sol	34.60	23.14	57.74	17,673,782	11.94
terra	18.30	24.56	42.86	12,642,019	4.82
luna	32.99	24.26	57.25	17,978,287	0.63
sonnet	62.97	39.31	102.28	57,599,306	18.88
opus	124.22	27.81	152.03	83,356,355	65.59
fable	55.94	26.04	81.98	16,901,363	49.70


## Takeaways
自己申告時間は開始・終了の切り方が異なるので、比較にはログのターン経過時間を採用。中断分を除いた最終指示のみの値もCSVに残している。

トークン数は読み直しを含む総処理量であり、文章のユニークな量ではない。各モデルのトークナイザーも異なる。コストは非キャッシュ入力・キャッシュ読出し・1時間キャッシュ書込・出力を別単価で計算する。

Jupyter関連ライブラリがこの環境にないため、全Pythonセルを同一の新しいPythonプロセスで上から順に実行し、出力を保存して検証した。Jupyterカーネル自体での実行は未確認。